# 01 — Error Level Analysis (ELA)

ELA reveals JPEG compression inconsistencies. Regions that were edited and re-saved at a different quality level will appear brighter in the ELA map than the surrounding (uniformly compressed) document.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

from src.ela import ELAAnalyzer
from src.utils import load_config

config = load_config('../config.yaml')
ela = ELAAnalyzer(config['ela'])
print('ELA config:', config['ela'])

## 1.1 — How ELA works

1. Re-save the original JPEG at a known lower quality.
2. Compute the pixel-wise absolute difference.
3. Amplify the difference for visibility.

Authentic images compress uniformly → low, consistent ELA values.  
Tampered regions have different compression history → higher ELA values.

In [ ]:
# ---- Put a sample image path here ----
sample_image = '../data/sample_images/sample.jpg'   # <-- replace

if not Path(sample_image).exists():
    # Create a synthetic example if no real sample is available
    import cv2
    img = np.ones((400, 600, 3), dtype=np.uint8) * 220
    cv2.putText(img, 'SAMPLE DOCUMENT', (80, 200),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (50, 50, 50), 3)
    Path('../data/sample_images').mkdir(parents=True, exist_ok=True)
    cv2.imwrite(sample_image, img)
    print('Created placeholder sample image')

fig = ela.visualize(sample_image)
plt.show()

score = ela.get_forgery_score(sample_image)
print(f'\nELA forgery score: {score}  (threshold = {config["ela"]["threshold"]})')
print(f'Verdict: {"SUSPICIOUS" if score > config["ela"]["threshold"] else "CLEAN"}')

## 1.2 — Comparison: authentic vs. forged

In [ ]:
import cv2
from src.data_generator import SyntheticForgeryGenerator

# Generate a quick synthetic forged version for comparison
gen = SyntheticForgeryGenerator(seed=0)

original = cv2.cvtColor(cv2.imread(sample_image), cv2.COLOR_BGR2RGB)
forged_arr, meta = gen.copy_move(original)

from PIL import Image
import io, tempfile, os

with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as f:
    forged_path = f.name
Image.fromarray(forged_arr).save(forged_path, quality=85)

ela_orig   = ela.analyze(sample_image)
ela_forged = ela.analyze(forged_path)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].imshow(original);    axes[0, 0].set_title('Original');     axes[0, 0].axis('off')
axes[0, 1].imshow(ela_orig);    axes[0, 1].set_title('ELA — Original'); axes[0, 1].axis('off')
axes[1, 0].imshow(forged_arr);  axes[1, 0].set_title('Forged (copy-move)'); axes[1, 0].axis('off')
axes[1, 1].imshow(ela_forged);  axes[1, 1].set_title('ELA — Forged');  axes[1, 1].axis('off')
plt.suptitle('ELA: Authentic vs. Copy-Move Forgery', fontsize=14)
plt.tight_layout()
plt.savefig('ela_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Authentic ELA score : {ela.get_forgery_score(sample_image)}')
print(f'Forged    ELA score : {ela.get_forgery_score(forged_path)}')

os.unlink(forged_path)

## 1.3 — ELA sensitivity to JPEG quality

In [ ]:
qualities = [70, 80, 85, 90, 95]
scores = []

for q in qualities:
    analyzer = ELAAnalyzer({'quality': q, 'amplification': 15})
    scores.append(analyzer.get_forgery_score(sample_image))

plt.figure(figsize=(8, 4))
plt.plot(qualities, scores, 'o-', linewidth=2, markersize=8)
plt.xlabel('Re-compression quality')
plt.ylabel('ELA forgery score')
plt.title('ELA Score vs. Re-compression Quality')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Quality 90 is a good balance — not too sensitive to normal compression, detects real edits.')

## 1.4 — Suspicious region localization

In [ ]:
regions = ela.get_suspicious_regions(sample_image)
print(f'Found {len(regions)} suspicious regions:')
for i, r in enumerate(regions[:5]):
    print(f'  [{i+1}] bbox={r["bbox"]}  area={r["area"]} px²')